# HealthBot: AI-Powered Patient Education System

**Project:** MediTech Solutions HealthBot Prototype  
**Technology Stack:** LangGraph + OpenAI + Tavily Search  
**Purpose:** Provide personalized, on-demand health information to patients

This notebook implements a conversational AI system that:
1. Searches for up-to-date medical information
2. Summarizes content in patient-friendly language
3. Assesses patient comprehension through quizzes
4. Provides educational feedback with citations

---

## 2. Install Required Libraries

Installing all necessary dependencies with specific versions for compatibility (in the virtual environment):

## 1. Create Virtual Environment

Setting up an isolated Python environment for the HealthBot project to avoid conflicts with your main Python installation:

In [1]:
# Create and activate virtual environment for HealthBot project
import subprocess
import sys
import os

# Create virtual environment
venv_name = "healthbot_venv"
print(f"🔧 Creating virtual environment '{venv_name}'...")

try:
    # Create virtual environment
    subprocess.run([sys.executable, "-m", "venv", venv_name], check=True)
    print(f"✅ Virtual environment '{venv_name}' created successfully!")
    
    # Determine the activation script path based on OS
    if os.name == 'nt':  # Windows
        activate_script = os.path.join(venv_name, "Scripts", "activate.ps1")
        python_exe = os.path.join(venv_name, "Scripts", "python.exe")
    else:  # Unix/Linux/MacOS
        activate_script = os.path.join(venv_name, "bin", "activate")
        python_exe = os.path.join(venv_name, "bin", "python")
    
    print(f"📍 Virtual environment created at: {os.path.abspath(venv_name)}")
    print(f"🐍 Python executable: {os.path.abspath(python_exe)}")
    print("\n" + "="*60)
    print("🔧 IMPORTANT: To activate this virtual environment manually:")
    print(f"   Windows PowerShell: .\\{activate_script}")
    print(f"   Windows CMD: {venv_name}\\Scripts\\activate.bat")
    print(f"   Unix/Linux/Mac: source {activate_script}")
    print("="*60)
    
    # Update sys.executable to use the virtual environment
    if os.path.exists(python_exe):
        sys.executable = os.path.abspath(python_exe)
        print(f"✅ Notebook will now use virtual environment Python: {sys.executable}")
    
except subprocess.CalledProcessError as e:
    print(f"❌ Error creating virtual environment: {e}")
    print("Please ensure you have python and venv module installed.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

🔧 Creating virtual environment 'healthbot_venv'...
✅ Virtual environment 'healthbot_venv' created successfully!
📍 Virtual environment created at: d:\healthCareAgent\healthbot_venv
🐍 Python executable: d:\healthCareAgent\healthbot_venv\Scripts\python.exe

🔧 IMPORTANT: To activate this virtual environment manually:
   Windows PowerShell: .\healthbot_venv\Scripts\activate.ps1
   Windows CMD: healthbot_venv\Scripts\activate.bat
   Unix/Linux/Mac: source healthbot_venv\Scripts\activate.ps1
✅ Notebook will now use virtual environment Python: d:\healthCareAgent\healthbot_venv\Scripts\python.exe


In [5]:
# Install required libraries for HealthBot prototype from requirements.txt
import subprocess
import sys
import os

# Verify installation by importing key modules
print("\n🧪 Testing imports...")
try:
    import langchain
    print("✅ langchain imported successfully")
    
    import langchain_openai
    print("✅ langchain_openai imported successfully")
    
    import langgraph
    print("✅ langgraph imported successfully")
    
    import tavily
    print("✅ tavily imported successfully")
    
    import dotenv
    print("✅ python-dotenv imported successfully")
    
    print("\n🎉 All key modules imported successfully!")
    print("✅ Environment is ready for HealthBot development!")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Some packages may not have installed correctly.")
    print("Please check the installation output above.")
except Exception as e:
    print(f"❌ Unexpected error during import testing: {e}")


🧪 Testing imports...
✅ langchain imported successfully
✅ langchain imported successfully
✅ langchain_openai imported successfully
✅ langgraph imported successfully
✅ tavily imported successfully
✅ python-dotenv imported successfully

🎉 All key modules imported successfully!
✅ Environment is ready for HealthBot development!
✅ langchain_openai imported successfully
✅ langgraph imported successfully
✅ tavily imported successfully
✅ python-dotenv imported successfully

🎉 All key modules imported successfully!
✅ Environment is ready for HealthBot development!


## 3. Load Environment Variables and API Keys

**Setup Instructions:**
1. **Tavily Account:** Sign up at [https://app.tavily.com/home](https://app.tavily.com/home) for free web search API (1000 requests free)
2. **OpenAI Account:** Get your API key from [https://platform.openai.com/api-keys](https://platform.openai.com/api-keys)
3. **Create config.env file** in the same directory as this notebook with:

```
OPENAI_API_KEY="sk-**********"
TAVILY_API_KEY="tvly-********"
```

In [6]:
# Load environment variables from config.env file
from dotenv import load_dotenv
import os

# Load the config.env file
load_dotenv('config.env')

# Verify that both API keys are loaded
try:
    assert os.getenv('OPENAI_API_KEY') is not None, "OPENAI_API_KEY not found in config.env"
    assert os.getenv('TAVILY_API_KEY') is not None, "TAVILY_API_KEY not found in config.env"
    
    print("✅ Environment variables loaded successfully!")
    print(f"✅ OpenAI API Key: {'*' * 20}...{os.getenv('OPENAI_API_KEY')[-4:]}")
    print(f"✅ Tavily API Key: {'*' * 20}...{os.getenv('TAVILY_API_KEY')[-4:]}")
    
except AssertionError as e:
    print(f"❌ Error: {e}")
    print("Please create a 'config.env' file with your API keys.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ Environment variables loaded successfully!
✅ OpenAI API Key: ********************...5008
✅ Tavily API Key: ********************...6Okb


## 4. Initialize LangChain Components

Setting up the core LangChain and LangGraph components for our workflow:

In [7]:
# Import core LangChain and LangGraph components
from langchain.schema import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

# LangGraph imports for workflow management
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict

# LangChain Hub for prompt templates
from langchain import hub

print("✅ LangChain and LangGraph components imported successfully!")
print("📋 Core components ready:")
print("   - StateGraph for workflow management")
print("   - Message handling for conversation flow")
print("   - Prompt templates for structured interactions")

✅ LangChain and LangGraph components imported successfully!
📋 Core components ready:
   - StateGraph for workflow management
   - Message handling for conversation flow
   - Prompt templates for structured interactions


## 5. Set Up Tavily Search Tool

Configuring the Tavily search tool for retrieving up-to-date medical information:

In [8]:
# Import and configure Tavily search tool
from langchain_community.tools.tavily_search import TavilySearchResults
from tavily import TavilyClient

# Initialize Tavily search tool
try:
    # Create Tavily search tool with medical-focused configuration
    tavily_tool = TavilySearchResults(
        max_results=5,  # Limit results for focused information
        search_depth="advanced",  # Get more comprehensive results
        include_answer=True,  # Include AI-generated answers
        include_raw_content=True,  # Include source content for citations
        api_key=os.getenv('TAVILY_API_KEY')
    )
    
    # Test client for direct API access if needed
    tavily_client = TavilyClient(api_key=os.getenv('TAVILY_API_KEY'))
    
    print("✅ Tavily search tool configured successfully!")
    print("🔍 Search capabilities:")
    print("   - Max 5 results per search for focused information")
    print("   - Advanced search depth for comprehensive coverage")
    print("   - Includes AI-generated answers and raw content")
    print("   - Optimized for medical/health information retrieval")
    
except Exception as e:
    print(f"❌ Error configuring Tavily: {e}")
    print("Please verify your TAVILY_API_KEY in config.env")

✅ Tavily search tool configured successfully!
🔍 Search capabilities:
   - Max 5 results per search for focused information
   - Advanced search depth for comprehensive coverage
   - Includes AI-generated answers and raw content
   - Optimized for medical/health information retrieval


C:\Users\kumarhalder\AppData\Local\Temp\ipykernel_64868\195798928.py:8: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


## 6. Configure OpenAI Language Model

Setting up the OpenAI language model for natural language understanding and generation:

In [11]:
# Import and configure OpenAI language model
from langchain_openai import ChatOpenAI

try:
    # Initialize OpenAI model with Vocareum configuration
    llm = ChatOpenAI(
        model="gpt-4o-mini",  # Cost-effective model good for prototyping
        temperature=0.3,  # Lower temperature for more consistent medical information
        max_tokens=1000,  # Adequate for detailed explanations
        base_url="https://openai.vocareum.com/v1",  # Vocareum base URL
        api_key=os.getenv('OPENAI_API_KEY')
    )
    
    print("✅ OpenAI language model configured successfully!")
    print("🤖 Model configuration:")
    print("   - Model: gpt-4o-mini (cost-effective for prototyping)")
    print("   - Temperature: 0.3 (consistent medical information)")
    print("   - Max tokens: 1000 (detailed explanations)")
    print("   - Base URL: https://openai.vocareum.com/v1 (Vocareum)")
    print("   - Optimized for healthcare education content")
    
except Exception as e:
    print(f"❌ Error configuring OpenAI: {e}")
    print("Please verify your OPENAI_API_KEY and base_url in config.env")

✅ OpenAI language model configured successfully!
🤖 Model configuration:
   - Model: gpt-4o-mini (cost-effective for prototyping)
   - Temperature: 0.3 (consistent medical information)
   - Max tokens: 1000 (detailed explanations)
   - Base URL: https://openai.vocareum.com/v1 (Vocareum)
   - Optimized for healthcare education content


## 7. Test API Connections

Verifying that both APIs are working correctly with simple test queries:

In [12]:
# Test OpenAI API connection
print("🧪 Testing OpenAI API connection...")
try:
    test_response = llm.invoke([HumanMessage(content="What is the capital of France? Answer in one word.")])
    print(f"✅ OpenAI API test successful! Response: {test_response.content}")
except Exception as e:
    print(f"❌ OpenAI API test failed: {e}")

print("\n" + "="*50)

# Test Tavily API connection
print("🧪 Testing Tavily API connection...")
try:
    test_search = tavily_tool.invoke({"query": "symptoms of common cold"})
    if test_search and len(test_search) > 0:
        print(f"✅ Tavily API test successful! Found {len(test_search)} results")
        print(f"   Sample result: {test_search[0].get('title', 'No title')[:50]}...")
    else:
        print("⚠️ Tavily API responded but returned no results")
except Exception as e:
    print(f"❌ Tavily API test failed: {e}")

print("\n" + "="*50)
print("🎉 API Connection Tests Complete!")
print("\nNext steps:")
print("1. ✅ APIs are configured and tested")
print("2. 🔄 Ready to implement HealthBot workflow")
print("3. 🏗️ Build LangGraph state machine")
print("4. 📚 Implement patient education pipeline")

🧪 Testing OpenAI API connection...
✅ OpenAI API test successful! Response: Paris.

🧪 Testing Tavily API connection...
✅ OpenAI API test successful! Response: Paris.

🧪 Testing Tavily API connection...
✅ Tavily API test successful! Found 5 results
   Sample result: Common Cold (Rhinovirus): Symptoms, Causes & Treat...

🎉 API Connection Tests Complete!

Next steps:
1. ✅ APIs are configured and tested
2. 🔄 Ready to implement HealthBot workflow
3. 🏗️ Build LangGraph state machine
4. 📚 Implement patient education pipeline
✅ Tavily API test successful! Found 5 results
   Sample result: Common Cold (Rhinovirus): Symptoms, Causes & Treat...

🎉 API Connection Tests Complete!

Next steps:
1. ✅ APIs are configured and tested
2. 🔄 Ready to implement HealthBot workflow
3. 🏗️ Build LangGraph state machine
4. 📚 Implement patient education pipeline


---

## ✅ Setup Complete!

**Status:** All components are now configured and ready for HealthBot development.

**What's Next:**
- **State Management:** Define the conversation state structure
- **Workflow Implementation:** Build the 7-phase LangGraph workflow
- **Patient Interface:** Create user-friendly conversation flow
- **Testing & Validation:** Ensure medical accuracy and usability

**Ready to proceed with HealthBot prototype development!** 🚀

---

# 🤖 HealthBot Workflow Implementation

Now that all components are configured, let's implement the 7-phase HealthBot workflow according to the project requirements:

1. **Topic Inquiry** - Ask patient for health topic
2. **Information Gathering** - Search with Tavily
3. **Information Processing** - Summarize in patient-friendly language
4. **Information Presentation** - Present to patient
5. **Comprehension Assessment** - Generate quiz question
6. **Response Evaluation** - Grade and provide feedback
7. **Session Management** - Continue or exit

## 8. Define HealthBot State Structure

Setting up the state management for our LangGraph workflow to track conversation flow and data:

In [13]:
# Define the state structure for HealthBot workflow
from typing import TypedDict, List, Optional
from dataclasses import dataclass

class HealthBotState(TypedDict):
    """State management for HealthBot conversation flow"""
    
    # Current conversation phase
    current_phase: str
    
    # Patient input and topic
    patient_query: str
    health_topic: str
    
    # Search and information processing
    search_results: List[dict]
    summarized_info: str
    information_sources: List[str]
    
    # Quiz and assessment
    quiz_question: str
    quiz_answer_options: List[str]
    correct_answer: str
    patient_answer: str
    
    # Evaluation and feedback
    grade: str
    explanation: str
    citations: List[str]
    
    # Session management
    session_active: bool
    continue_learning: Optional[bool]
    
    # Error handling
    error_message: Optional[str]

# Initialize state
def create_initial_state() -> HealthBotState:
    """Create the initial state for a new HealthBot session"""
    return HealthBotState(
        current_phase="topic_inquiry",
        patient_query="",
        health_topic="",
        search_results=[],
        summarized_info="",
        information_sources=[],
        quiz_question="",
        quiz_answer_options=[],
        correct_answer="",
        patient_answer="",
        grade="",
        explanation="",
        citations=[],
        session_active=True,
        continue_learning=None,
        error_message=None
    )

print("✅ HealthBot State structure defined successfully!")
print("📋 State includes:")
print("   - Conversation phase tracking")
print("   - Patient input and topic management")
print("   - Search results and information processing")
print("   - Quiz generation and assessment")
print("   - Evaluation and feedback system")
print("   - Session management and error handling")

✅ HealthBot State structure defined successfully!
📋 State includes:
   - Conversation phase tracking
   - Patient input and topic management
   - Search results and information processing
   - Quiz generation and assessment
   - Evaluation and feedback system
   - Session management and error handling


## 9. Implement Workflow Node Functions

Creating the individual functions for each phase of the HealthBot workflow:

In [14]:
# Phase 1: Topic Inquiry Node
def topic_inquiry_node(state: HealthBotState) -> HealthBotState:
    """
    Ask the patient what health topic they'd like to learn about
    """
    print("=" * 60)
    print("🏥 Welcome to HealthBot - Your AI Health Education Assistant!")
    print("=" * 60)
    print("\nI'm here to help you learn about health topics in a friendly,")
    print("easy-to-understand way. After providing information, I'll test")
    print("your understanding with a simple quiz question.\n")
    
    # Get patient input
    patient_query = input("💬 What health topic or medical condition would you like to learn about? ")
    
    if not patient_query.strip():
        state["error_message"] = "Please provide a health topic to learn about."
        return state
    
    # Process and validate the query
    state["patient_query"] = patient_query.strip()
    state["health_topic"] = patient_query.strip()
    state["current_phase"] = "information_gathering"
    state["error_message"] = None
    
    print(f"\n✅ Great! I'll help you learn about: {state['health_topic']}")
    print("🔍 Let me search for the most current and reliable information...\n")
    
    return state

print("✅ Phase 1: Topic Inquiry function created!")

✅ Phase 1: Topic Inquiry function created!


In [15]:
# Phase 2: Information Gathering Node
def information_gathering_node(state: HealthBotState) -> HealthBotState:
    """
    Use Tavily search to find relevant, up-to-date medical information
    """
    try:
        health_topic = state["health_topic"]
        
        # Create a medical-focused search query
        search_query = f"medical information about {health_topic} symptoms causes treatment prevention"
        
        print(f"🔍 Searching for medical information about: {health_topic}")
        
        # Search using Tavily
        search_results = tavily_tool.invoke({"query": search_query})
        
        if not search_results:
            state["error_message"] = "No search results found. Please try a different health topic."
            return state
            
        # Store results and extract sources
        state["search_results"] = search_results
        sources = []
        
        for result in search_results:
            if 'url' in result:
                sources.append(result['url'])
        
        state["information_sources"] = sources
        state["current_phase"] = "information_processing"
        
        print(f"✅ Found {len(search_results)} relevant medical sources")
        print("📚 Processing information to create patient-friendly summary...\n")
        
    except Exception as e:
        state["error_message"] = f"Error during information gathering: {str(e)}"
        print(f"❌ Search error: {e}")
    
    return state

print("✅ Phase 2: Information Gathering function created!")

✅ Phase 2: Information Gathering function created!


In [16]:
# Phase 3: Information Processing Node
def information_processing_node(state: HealthBotState) -> HealthBotState:
    """
    Summarize search results into patient-friendly language
    """
    try:
        search_results = state["search_results"]
        health_topic = state["health_topic"]
        
        # Combine search results content
        combined_content = ""
        for result in search_results:
            if 'content' in result:
                combined_content += result['content'] + "\n\n"
            elif 'answer' in result:
                combined_content += result['answer'] + "\n\n"
        
        # Create summarization prompt
        summarization_prompt = f"""
        You are a healthcare educator. Create a clear, patient-friendly summary about {health_topic}.
        
        Use this medical information:
        {combined_content}
        
        Please provide a comprehensive but easy-to-understand summary that includes:
        1. What it is (definition/overview)
        2. Main symptoms or signs
        3. Common causes
        4. Treatment options
        5. Prevention tips (if applicable)
        
        Use simple language, avoid medical jargon, and structure the information clearly.
        Keep it informative but accessible to patients without medical background.
        Limit to approximately 300-400 words.
        """
        
        print("🧠 Creating patient-friendly summary using AI...")
        
        # Generate summary using LLM
        response = llm.invoke([HumanMessage(content=summarization_prompt)])
        
        state["summarized_info"] = response.content
        state["current_phase"] = "information_presentation"
        
        print("✅ Medical information processed into patient-friendly format")
        
    except Exception as e:
        state["error_message"] = f"Error during information processing: {str(e)}"
        print(f"❌ Processing error: {e}")
    
    return state

print("✅ Phase 3: Information Processing function created!")

✅ Phase 3: Information Processing function created!


In [17]:
# Phase 4: Information Presentation Node
def information_presentation_node(state: HealthBotState) -> HealthBotState:
    """
    Present the summarized information to the patient
    """
    print("=" * 60)
    print(f"📚 HEALTH INFORMATION: {state['health_topic'].upper()}")
    print("=" * 60)
    
    # Present the summarized information
    print(state["summarized_info"])
    
    print("\n" + "=" * 60)
    print("📖 Please take your time to read and understand this information.")
    print("When you're ready, I'll ask you a question to check your understanding.")
    print("=" * 60)
    
    # Wait for patient confirmation
    ready = input("\n✅ Press Enter when you're ready for the comprehension check... ")
    
    state["current_phase"] = "comprehension_assessment"
    
    print("\n🧪 Generating a comprehension question based on the information...\n")
    
    return state

print("✅ Phase 4: Information Presentation function created!")

✅ Phase 4: Information Presentation function created!


In [18]:
# Phase 5: Comprehension Assessment Node
def comprehension_assessment_node(state: HealthBotState) -> HealthBotState:
    """
    Generate a relevant quiz question based on the provided information
    """
    try:
        summarized_info = state["summarized_info"]
        health_topic = state["health_topic"]
        
        # Create quiz generation prompt
        quiz_prompt = f"""
        Based on this health information about {health_topic}:
        
        {summarized_info}
        
        Create a single, relevant multiple-choice question to test patient understanding.
        
        Format your response as:
        QUESTION: [Your question here]
        A) [Option A]
        B) [Option B] 
        C) [Option C]
        D) [Option D]
        CORRECT: [Letter of correct answer]
        EXPLANATION: [Brief explanation of why this is correct]
        
        Make the question practical and focused on key information patients should remember.
        Ensure options are clearly distinct and only one is definitively correct.
        """
        
        print("🧠 Generating comprehension question...")
        
        # Generate quiz using LLM
        response = llm.invoke([HumanMessage(content=quiz_prompt)])
        quiz_content = response.content
        
        # Parse the response to extract components
        lines = quiz_content.split('\n')
        question = ""
        options = []
        correct_answer = ""
        explanation = ""
        
        for line in lines:
            line = line.strip()
            if line.startswith("QUESTION:"):
                question = line.replace("QUESTION:", "").strip()
            elif line.startswith(("A)", "B)", "C)", "D)")):
                options.append(line)
            elif line.startswith("CORRECT:"):
                correct_answer = line.replace("CORRECT:", "").strip()
            elif line.startswith("EXPLANATION:"):
                explanation = line.replace("EXPLANATION:", "").strip()
        
        # Store quiz information
        state["quiz_question"] = question
        state["quiz_answer_options"] = options
        state["correct_answer"] = correct_answer
        state["explanation"] = explanation
        state["current_phase"] = "response_collection"
        
        # Present the quiz question
        print("=" * 60)
        print("🧪 COMPREHENSION CHECK")
        print("=" * 60)
        print(f"\n❓ {question}\n")
        
        for option in options:
            print(f"   {option}")
        
        print("\n" + "=" * 60)
        
    except Exception as e:
        state["error_message"] = f"Error during quiz generation: {str(e)}"
        print(f"❌ Quiz generation error: {e}")
    
    return state

print("✅ Phase 5: Comprehension Assessment function created!")

✅ Phase 5: Comprehension Assessment function created!


In [19]:
# Phase 6: Response Evaluation Node
def response_evaluation_node(state: HealthBotState) -> HealthBotState:
    """
    Collect patient answer, evaluate it, and provide feedback
    """
    try:
        # Get patient's answer
        patient_answer = input("💬 Your answer (A, B, C, or D): ").strip().upper()
        
        if patient_answer not in ['A', 'B', 'C', 'D']:
            print("⚠️ Please enter A, B, C, or D")
            return state
        
        state["patient_answer"] = patient_answer
        correct_answer = state["correct_answer"].upper()
        
        # Evaluate the response
        if patient_answer == correct_answer:
            grade = "✅ CORRECT!"
            print(f"\n🎉 {grade}")
        else:
            grade = "❌ INCORRECT"
            print(f"\n😔 {grade}")
            print(f"The correct answer was: {correct_answer}")
        
        state["grade"] = grade
        
        # Provide detailed explanation with citations
        print("\n" + "=" * 60)
        print("📝 EXPLANATION")
        print("=" * 60)
        print(f"\n{state['explanation']}")
        
        # Add citations from sources
        if state["information_sources"]:
            print(f"\n📚 This information was gathered from {len(state['information_sources'])} reliable medical sources:")
            for i, source in enumerate(state["information_sources"][:3], 1):  # Show top 3 sources
                print(f"   {i}. {source}")
        
        print("\n" + "=" * 60)
        
        state["current_phase"] = "session_management"
        
    except Exception as e:
        state["error_message"] = f"Error during response evaluation: {str(e)}"
        print(f"❌ Evaluation error: {e}")
    
    return state

print("✅ Phase 6: Response Evaluation function created!")

✅ Phase 6: Response Evaluation function created!


In [20]:
# Phase 7: Session Management Node
def session_management_node(state: HealthBotState) -> HealthBotState:
    """
    Ask if patient wants to learn about another topic or exit
    """
    print("\n" + "=" * 60)
    print("🎯 LEARNING SESSION COMPLETE!")
    print("=" * 60)
    
    # Ask about continuing
    continue_choice = input("\n🤔 Would you like to learn about another health topic? (yes/no): ").strip().lower()
    
    if continue_choice in ['yes', 'y', 'yeah', 'sure', 'ok', 'okay']:
        print("\n🔄 Starting a new learning session...")
        print("📝 Resetting for privacy and accuracy...\n")
        
        # Reset state for new topic (privacy and accuracy)
        new_state = create_initial_state()
        new_state["current_phase"] = "topic_inquiry"
        return new_state
        
    else:
        print("\n👋 Thank you for using HealthBot!")
        print("💙 Remember: This information is for educational purposes.")
        print("🏥 Always consult healthcare professionals for medical advice.")
        print("=" * 60)
        
        state["session_active"] = False
        state["continue_learning"] = False
        state["current_phase"] = "session_complete"
    
    return state

print("✅ Phase 7: Session Management function created!")

✅ Phase 7: Session Management function created!


## 10. Build LangGraph Workflow

Now let's create the LangGraph state machine that orchestrates the entire HealthBot workflow:

In [26]:
# Build the LangGraph workflow
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
def build_healthbot_workflow():
    """
    Create the LangGraph workflow for HealthBot
    """
    # Create the state graph
    workflow = StateGraph(HealthBotState)
    
    # Add all workflow nodes
    workflow.add_node("topic_inquiry", topic_inquiry_node)
    workflow.add_node("information_gathering", information_gathering_node)
    workflow.add_node("information_processing", information_processing_node)
    workflow.add_node("information_presentation", information_presentation_node)
    workflow.add_node("comprehension_assessment", comprehension_assessment_node)
    workflow.add_node("response_evaluation", response_evaluation_node)
    workflow.add_node("session_management", session_management_node)
    
    # Define the workflow edges (transitions between phases)
    workflow.add_edge(START, "topic_inquiry")
    workflow.add_edge("topic_inquiry", "information_gathering")
    workflow.add_edge("information_gathering", "information_processing")
    workflow.add_edge("information_processing", "information_presentation")
    workflow.add_edge("information_presentation", "comprehension_assessment")
    workflow.add_edge("comprehension_assessment", "response_evaluation")
    workflow.add_edge("response_evaluation", "session_management")
    
    # Conditional edge for session management
    def should_continue(state: HealthBotState) -> str:
        """Determine if we should continue or end the session"""
        if state.get("current_phase") == "topic_inquiry" and state.get("session_active", True):
            return "topic_inquiry"
        elif state.get("session_active", True) and state.get("current_phase") != "session_complete":
            return "topic_inquiry"
        else:
            return END
    
    workflow.add_conditional_edges(
        "session_management",
        should_continue,
        {
            "topic_inquiry": "topic_inquiry",
            END: END
        }
    )
    
    # Compile the workflow
    app = workflow.compile()
    return app

# Build the HealthBot application
healthbot_app = build_healthbot_workflow()

print("✅ HealthBot LangGraph workflow created successfully!")
print("🔄 Workflow includes:")
print("   1. Topic Inquiry → 2. Information Gathering → 3. Information Processing")
print("   4. Information Presentation → 5. Comprehension Assessment → 6. Response Evaluation")
print("   7. Session Management (with loop-back or exit)")
print("\n🤖 HealthBot is ready to educate patients!")

# Visualize the workflow graph
print("\n📊 Generating workflow visualization...")
try:
    from IPython.display import Image, display
    
    # Generate the graph image
    graph_image = healthbot_app.get_graph().draw_mermaid_png()
    
    # Display the graph
    display(Image(graph_image))
    print("✅ HealthBot workflow diagram displayed above!")
    
except Exception as e:
    print(f"⚠️ Could not generate graph visualization: {e}")
    print("📝 The workflow is still functional - visualization is optional")
    print("💡 You may need to install: pip install pygraphviz or graphviz")

✅ HealthBot LangGraph workflow created successfully!
🔄 Workflow includes:
   1. Topic Inquiry → 2. Information Gathering → 3. Information Processing
   4. Information Presentation → 5. Comprehension Assessment → 6. Response Evaluation
   7. Session Management (with loop-back or exit)

🤖 HealthBot is ready to educate patients!

📊 Generating workflow visualization...
⚠️ Could not generate graph visualization: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 502.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
📝 The workflow is still functional - visualization is optional
💡 You may need to install: pip install pygraphviz or graphviz


## 11. Run HealthBot Interactive Session

Let's start the HealthBot and interact with a patient! The system will guide them through the complete learning and assessment cycle:

In [27]:
# Run the HealthBot Interactive Session
def run_healthbot():
    """
    Main function to run the HealthBot patient education system
    """
    print("🤖 Initializing HealthBot AI-Powered Patient Education System...")
    print("🏥 MediTech Solutions - Healthcare Innovation")
    
    try:
        # Create initial state
        initial_state = create_initial_state()
        
        # Run the workflow
        final_state = healthbot_app.invoke(initial_state)
        
        print("\n✅ HealthBot session completed successfully!")
        return final_state
        
    except KeyboardInterrupt:
        print("\n\n⚠️ Session interrupted by user.")
        print("👋 Thank you for using HealthBot!")
        
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")
        print("🔧 Please check your API keys and internet connection.")

# Start the HealthBot!
print("🚀 Ready to start HealthBot patient education session!")
print("📋 The system will:")
print("   • Ask for a health topic")
print("   • Search for current medical information")  
print("   • Provide patient-friendly education")
print("   • Test comprehension with a quiz")
print("   • Give detailed feedback and citations")
print("   • Offer to learn about additional topics")
print("\n" + "="*60)
print("Click 'Run' on this cell to start your HealthBot session!")
print("="*60)

# Uncomment the line below to auto-start HealthBot
run_healthbot()

🚀 Ready to start HealthBot patient education session!
📋 The system will:
   • Ask for a health topic
   • Search for current medical information
   • Provide patient-friendly education
   • Test comprehension with a quiz
   • Give detailed feedback and citations
   • Offer to learn about additional topics

Click 'Run' on this cell to start your HealthBot session!
🤖 Initializing HealthBot AI-Powered Patient Education System...
🏥 MediTech Solutions - Healthcare Innovation
🏥 Welcome to HealthBot - Your AI Health Education Assistant!

I'm here to help you learn about health topics in a friendly,
easy-to-understand way. After providing information, I'll test
your understanding with a simple quiz question.


✅ Great! I'll help you learn about: diarhoea
🔍 Let me search for the most current and reliable information...

🔍 Searching for medical information about: diarhoea
✅ Found 5 relevant medical sources
📚 Processing information to create patient-friendly summary...

🧠 Creating patient-friendl

{'current_phase': 'session_complete',
 'patient_query': 'diarhoea',
 'health_topic': 'diarhoea',
 'search_results': [{'title': 'Diarrhea: Causes, Symptoms & Treatment - Cleveland Clinic',
   'url': 'https://my.clevelandclinic.org/health/diseases/4108-diarrhea',
   'content': 'Diarrhea # Diarrhea ### What is diarrhea? #### How common is diarrhea? ### What causes diarrhea? Diarrhea is a common medication side effect. Diarrhea is a common symptom of conditions that cause irritation and inflammation in your bowels (intestines). ### What are the symptoms of diarrhea? Severe cases of diarrhea may signal a medical condition, like a serious infection, that won’t get better without treatment from a healthcare provider. Contact your provider if you have diarrhea with: ### Can diarrhea be prevented? Call your healthcare provider if you have diarrhea that doesn’t improve or go away within a few days. If it doesn’t, or if you’re experiencing symptoms of severe diarrhea, contact your healthcare prov

---

## 🎉 HealthBot Prototype Complete!

### ✅ **What We've Built:**

The **HealthBot AI-Powered Patient Education System** is now fully implemented according to MediTech Solutions' requirements:

#### **🔄 7-Phase Workflow:**
1. **Topic Inquiry** - Asks patients about desired health topics
2. **Information Gathering** - Uses Tavily search for current medical information
3. **Information Processing** - AI summarizes content in patient-friendly language
4. **Information Presentation** - Displays structured, accessible health information
5. **Comprehension Assessment** - Generates relevant quiz questions
6. **Response Evaluation** - Grades answers and provides detailed feedback
7. **Session Management** - Offers new topics or session completion with state reset

#### **🏗️ Technical Implementation:**
- **LangGraph State Machine** - Orchestrates the entire workflow
- **OpenAI Integration** - Powers natural language understanding and generation
- **Tavily Search** - Provides up-to-date medical information retrieval
- **State Management** - Tracks conversation flow and maintains data privacy
- **Error Handling** - Graceful handling of API failures and user inputs

#### **🎯 Business Goals Achieved:**
- ✅ **24/7 Access** to reliable health information
- ✅ **Patient-Friendly** medical information presentation
- ✅ **Comprehension Testing** to ensure understanding
- ✅ **Citation Support** for information credibility
- ✅ **Privacy Protection** through state reset between topics
- ✅ **Scalable Architecture** ready for production deployment

### 🚀 **Ready for Testing:**
Run the final cell above to start an interactive HealthBot session and test the complete patient education workflow!

### 📈 **Future Enhancements:**
- Multi-language support
- Voice interaction capabilities
- Integration with Electronic Health Records (EHR)
- Advanced analytics and learning metrics
- Specialized medical domain versions